In [ ]:
# RAG 예제: "백과사전 질문-답변기

!pip install openai sentence-transformers chromadb python-dotenv


In [ ]:
from openai import OpenAI
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import os
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
model = "gpt-4o-mini"

# 1. 지식 구성 ----------------------------
# RAG에서 검색 대상이 되는 간단한 지식 문서들
knowledge = [
    "기린은 목이 길다.",
    "코끼리는 귀가 크고 무겁다.",
    "치타는 육상에서 가장 빠른 동물이다.",
    "하마는 물속에서 생활하는 포유류다.",
    "펭귄은 날지 못하지만 수영을 잘한다."
]


In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")  # 2. 임베딩 모델 로드

# 지식 문서들을 임베딩 벡터로 변환
# normalize_embeddings=True를 사용하면 벡터 크기가 1로 정규화됨
# 정규화된 벡터끼리 내적하면 cosine similarity로 해석하기 쉬움
embeddings = embedder.encode(
    knowledge,
    normalize_embeddings=True
)
print("문서 임베딩:", embeddings)


In [ ]:
# 3. Chroma 저장소 구성 ----------------------------------
# ChromaDB를 로컬 디스크에 저장하는 벡터 데이터베이스 클라이언트 생성
chroma_client = chromadb.Client(Settings(
    persist_directory="./rag_demo",
    anonymized_telemetry=False   # 익명 사용 정보 전송 비활성화(권장)
))

# 실습에서는 같은 ID가 중복 추가될 수 있으므로 기존 컬렉션을 삭제하고 다시 생성
try:
    chroma_client.delete_collection("animals")
except:
    pass

# cosine 거리 기준으로 animals 컬렉션 생성
collection = chroma_client.create_collection(
    name="animals",
    metadata={"hnsw:space": "cosine"}
)

# 문서와 임베딩 벡터를 ChromaDB에 저장
for i, (text, emb) in enumerate(zip(knowledge, embeddings)):
    collection.add(
        documents=[text],
        embeddings=[emb.tolist()],
        ids=[f"doc_{i}"]
    )

# 저장된 문서와 ID 확인
all_data = collection.get()
print("\n저장된 전체 데이터:", all_data)
print("\n컬렉션에 저장된 문서 개수:", collection.count())

# 특정 ID의 문서만 조회
doc = collection.get(ids=["doc_0"])
print("\n특정 문서 조회:", doc)


In [ ]:
# 4. 질문 처리 ---------
query = "목이 긴 동물은?"    # 사용자의 질문

# 질문을 임베딩 벡터로 변환
query_vec = embedder.encode(
    [query],
    normalize_embeddings=True
)[0]

# 질문 벡터와 가까운 문서를 ChromaDB에서 검색
results = collection.query(
    query_embeddings=[query_vec.tolist()],
    n_results=3,
    include=["documents", "distances"]
)
print("\n검색 결과:", results)
# 검색된 문서들을 하나의 참고 문맥으로 합치기
context = "\n".join(results["documents"][0])
print("\n검색된 참고 문서:", context)

In [ ]:
# 5. GPT에 연결하여 답변 생성 --------------------------
# 검색된 문서와 사용자 질문을 함께 프롬프트에 넣음
# 이것이 RAG에서 Generation 단계로 넘어가는 핵심 부분
prompt = f"""
아래 정보를 참고해서 질문에 답해줘.
질문 정보를 바탕으로 질문에 대해 상세하게 설명해 줘.
가능하면 예시나 관련 배경지식도 함께 덧붙여.

[참고 정보]
{context}

[질문]
{query}

추가 사항:
마크다운 기호 없이 평문으로 답해줘.
"""

print("프롬프트 확인 ---")
print(prompt)

# GPT 모델에 프롬프트를 전달하여 답변 생성
response = client.responses.create(
    model=model,  input=prompt
)
print("\n답변:", response.output_text)
